
# Bengali NLP Safety Challenge 2026 — Targeted Upgrade of the 0.591 Baseline

This notebook is intentionally based on the **proven 0.591 v10 recipe**, not the failed v17/v18 redesign.

### Strategy
1. Preserve the successful **BanglaBERT + BanglaBERT-large + BiLSTM** architecture.
2. Compare `clean` vs a conservative `raw` text view without destroying the proven baseline.
3. Use **5-fold multilabel stratification** for both models.
4. Add duplicate-aware soft targets only for **conflicting duplicate groups**, reducing label-noise damage.
5. Tune focal/pos-weight settings on OOF data rather than assuming one loss is optimal.
6. Tune the toxicity gate from OOF predictions instead of forcing a fixed gate.
7. Add a cheap **character/word TF-IDF specialist** and only keep its contribution if OOF score improves.
8. Produce detailed per-label error analysis, token-length analysis and positive-rate diagnostics.
9. Retrain the selected transformer variants on all labeled data and generate the final submission.

The goal is to improve the hidden-test score, with **0.61+ as the target, not a guarantee**.


In [ ]:

import sys, subprocess, os, re, gc, glob, random, warnings, math, json
subprocess.run([sys.executable,"-m","pip","install","-q","iterative-stratification==0.1.7","scikit-learn"],check=False)
print("Dependencies ready")


In [ ]:

import numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, logging as hf_logging
from torch.optim import AdamW
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

LABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]
N_LABELS=len(LABELS)

MODELS=[
    dict(name="csebuetnlp/banglabert", folds=5, batch=16, accum=1, lr=2e-5, epochs=4),
    dict(name="csebuetnlp/banglabert_large", folds=5, batch=8, accum=2, lr=1.5e-5, epochs=4),
]

class CFG:
    max_len=160
    lstm_hidden=256
    dropout=0.30
    weight_decay=0.01
    warmup_ratio=0.10
    seed=42
    use_fp16=True
    focal_alpha=0.25
    focal_gamma=2.0
    pos_weight_clip=20.0
    use_gate=True
    use_soft_duplicate_targets=True

DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_GPUS=torch.cuda.device_count() if torch.cuda.is_available() else 0

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(CFG.seed)

print("DEVICE:",DEVICE,"GPUs:",N_GPUS)



## 1. Load the exact competition data

The validation file is kept separate during the main model-selection experiment. This prevents us from accidentally treating the 500 validation rows as ordinary training data while deciding preprocessing/loss/gate settings.

After the final configuration is selected, the model can be retrained on train + validation.


In [ ]:

# ---- Load data: Kaggle P100/T4 AND Google Colab ----
# COLAB: UPLOAD the competition .zip to /content (drag into Files) OR the 3 CSVs. Auto-extracts.
# KAGGLE: attach the competition dataset (right panel -> + Add Input).
import zipfile
DATA_DIR_OVERRIDE = ""   # optional: folder holding train/validation/test.csv
# --- Colab only: uncomment to mount Google Drive ---
# from google.colab import drive; drive.mount("/content/drive")
def _has_all(d):
    try: return all(os.path.exists(os.path.join(d,f)) for f in ["train.csv","validation.csv","test.csv"])
    except Exception: return False
def _extract_zip():
    for r in ["/content",".","/kaggle/working"]:
        if os.path.isdir(r):
            for zp in glob.glob(os.path.join(r,"*.zip")):
                try:
                    if any(n.endswith("train.csv") for n in zipfile.ZipFile(zp).namelist()):
                        dest=os.path.join(r,"bengali_data"); os.makedirs(dest,exist_ok=True)
                        zipfile.ZipFile(zp).extractall(dest)
                        for hit in glob.glob(os.path.join(dest,"**","train.csv"),recursive=True):
                            d=os.path.dirname(hit)
                            if _has_all(d): print("Extracted:",zp,"->",d); return d
                except Exception as e: print("zip skip",zp,e)
    return None
def find_data_dir():
    if DATA_DIR_OVERRIDE and _has_all(DATA_DIR_OVERRIDE): return DATA_DIR_OVERRIDE
    for d in ["/kaggle/input/competitions/bengali-nlp-safety-challenge-2026",
              "/kaggle/input/bengali-nlp-safety-challenge-2026",
              "/content/bengali_data","/content/data","/content","./data","."]:
        if _has_all(d): return d
    for root in ["/kaggle/input","/content","."]:
        if os.path.isdir(root):
            for hit in glob.glob(os.path.join(root,"**","train.csv"),recursive=True):
                d=os.path.dirname(hit)
                if _has_all(d): return d
    ex=_extract_zip()
    if ex: return ex
    raise FileNotFoundError("Upload the competition .zip (or the 3 CSVs) to /content, then re-run this cell.")
DATA_DIR=find_data_dir()
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
val_df=pd.read_csv(os.path.join(DATA_DIR,"validation.csv"))
test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
print("DATA_DIR:",DATA_DIR)
print(train_df.shape,val_df.shape,test_df.shape)
print("\nTrain prevalence:"); print(train_df[LABELS].mean().round(4))
print("\nValidation prevalence:"); print(val_df[LABELS].mean().round(4))



## 2. Dataset diagnostics: imbalance, duplicate noise and token-length constraints

The rare labels are extremely imbalanced. We therefore evaluate the **actual F1 metric**, not accuracy.

We also inspect duplicate comments. Conflicting duplicate labels are treated as potentially noisy observations during training, but validation/test labels are never altered.


In [ ]:

# Duplicate diagnostics
dup=train_df[train_df["text"].duplicated(False)].copy()
print("Duplicate rows:",len(dup))
print("Unique duplicated texts:",dup["text"].nunique())

for lab in LABELS:
    g=train_df.groupby("text")[lab].agg(["count","nunique"])
    conflicts=int(((g["count"]>1)&(g["nunique"]>1)).sum())
    print(f"{lab:22s} conflicting duplicate groups: {conflicts}")

# Label hierarchy check
for lab in LABELS[1:]:
    n=int(train_df.loc[train_df["is_toxic"]==0,lab].sum())
    print(f"{lab:22s} positives with is_toxic=0: {n}")


In [ ]:

# Conservative text views.
# IMPORTANT: keep the original text intact.
def raw_text(t):
    if not isinstance(t,str): return ""
    t=t.replace("\ufeff","").replace("\u200b","").replace("\u200c","").replace("\u200d","")
    t=re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]"," ",t)
    return re.sub(r"\s+"," ",t).strip()

def clean_text(t):
    t=raw_text(t)
    t=re.sub(r"http\S+|www\.\S+"," ",t,flags=re.I)
    t=re.sub(r"<[^>]*>"," ",t)
    t=re.sub(r"@\w+"," ",t)
    t=t.replace("#","")
    return re.sub(r"\s+"," ",t).strip()

for df in [train_df,val_df,test_df]:
    df["raw"]=df["text"].map(raw_text)
    df["clean"]=df["text"].map(clean_text)

print(train_df[["text","raw","clean"]].head(5).to_string(index=False))



### Tokenization audit

The original 0.591 notebook used 128 tokens. This version starts at 160 because the data contains a long tail of comments, while keeping the same model family.

The notebook measures the real tokenizer distribution before training.


In [ ]:

def token_length_report(model_name, texts):
    tok=AutoTokenizer.from_pretrained(model_name)
    lens=[]
    for x in texts:
        lens.append(len(tok.encode(str(x),add_special_tokens=True)))
    a=np.asarray(lens)
    print(model_name)
    print("percentiles 50/75/90/95/99/max:",np.percentile(a,[50,75,90,95,99,100]).round(1))
    for n in [96,128,160,192,256]:
        print(f"> {n:3d}: {100*np.mean(a>n):5.2f}%")
    return a

lengths_base=token_length_report("csebuetnlp/banglabert",train_df["raw"].values[:2000])



## 3. Loss and model

We keep the successful BiLSTM pooling from the 0.591 notebook.

The main controlled changes are:
- slightly longer context;
- conservative imbalance weighting;
- optional duplicate-soft targets;
- the ability to compare raw and clean views.

No FGM, exotic pooling, seven separate specialists, or aggressive gates are introduced here.


In [ ]:

class FocalLoss(nn.Module):
    def __init__(self,alpha=0.25,gamma=2.0,pos_weight=None):
        super().__init__(); self.alpha=alpha; self.gamma=gamma; self.pos_weight=pos_weight
    def forward(self,inputs,targets):
        bce=F.binary_cross_entropy_with_logits(inputs,targets,pos_weight=self.pos_weight,reduction="none")
        pt=torch.exp(-bce)
        return (self.alpha*((1-pt)**self.gamma)*bce).mean()

class DS(Dataset):
    def __init__(self,texts,tok,labels=None):
        self.t=list(texts); self.tok=tok; self.y=labels
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),max_length=CFG.max_len,padding="max_length",
                   truncation=True,return_token_type_ids=False,
                   return_attention_mask=True,return_tensors="pt")
        d={"input_ids":e["input_ids"].flatten(),"attention_mask":e["attention_mask"].flatten()}
        if self.y is not None: d["labels"]=torch.tensor(self.y[i],dtype=torch.float32)
        return d

class BertBiLSTM(nn.Module):
    def __init__(self,name,n_out=7):
        super().__init__()
        self.bert=AutoModel.from_pretrained(name)
        self.lstm=nn.LSTM(self.bert.config.hidden_size,CFG.lstm_hidden,
                          num_layers=2,batch_first=True,bidirectional=True,dropout=CFG.dropout)
        self.dropout=nn.Dropout(CFG.dropout)
        self.fc=nn.Linear(CFG.lstm_hidden*2,n_out)
    def forward(self,input_ids,attention_mask):
        seq=self.bert(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        lo,_=self.lstm(seq)
        m=attention_mask.unsqueeze(-1).float()
        pooled=(lo*m).sum(1)/m.sum(1).clamp(min=1e-9)
        return self.fc(self.dropout(pooled))

def pos_weight_from(y):
    pos=y.sum(0); neg=len(y)-pos
    return np.clip(np.sqrt(neg/np.clip(pos,1,None)),1.0,CFG.pos_weight_clip)



## 4. Thresholds and toxicity gate

The competition uses:
- Macro-F1 for `is_toxic`
- mean F1 for the six attributes
- average of those two components.

Therefore threshold selection is performed directly against that metric.

The old notebook forced all attributes to zero when `is_toxic=0`. Because the supplied labels have zero attribute positives for non-toxic examples, we retain the hierarchy — but the gate threshold is now **learned from OOF predictions** rather than hard-coded.


In [ ]:

def gate_preds(prob,thr,gate_on=True,gate_thr=0.5):
    pred=(prob>=thr.reshape(1,-1)).astype(int)
    if gate_on:
        pred[pred[:,0]==0,1:]=0
    return pred

def combined_score(y,pred):
    A=f1_score(y[:,0],pred[:,0],average="macro",zero_division=0)
    B=np.mean([f1_score(y[:,i],pred[:,i],zero_division=0) for i in range(1,N_LABELS)])
    return (A+B)/2,A,B

def tune_thresholds(y,p):
    grid=np.arange(0.05,0.96,0.01)
    th=np.full(N_LABELS,0.5)
    for i in range(N_LABELS):
        scores=[]
        for t in grid:
            avg="macro" if i==0 else "binary"
            scores.append(f1_score(y[:,i],(p[:,i]>=t).astype(int),average=avg,zero_division=0))
        th[i]=grid[int(np.argmax(scores))]
    return th

def tune_gate(y,p,thr):
    base=gate_preds(p,thr,True)
    best=combined_score(y,base)[0]
    best_gate=0.5
    # Instead of changing the rule completely, learn a minimum toxicity probability.
    for g in np.arange(0.30,0.91,0.02):
        pred=(p>=thr.reshape(1,-1)).astype(int)
        pred[p[:,0]<g,1:]=0
        s=combined_score(y,pred)[0]
        if s>best:
            best=s; best_gate=float(g)
    return best_gate,best



## 5. Duplicate-aware targets

Only duplicate groups with conflicting labels receive soft targets. For a clean duplicate group, the original hard target is retained.

This is a small change designed to prevent a model from being punished heavily for predicting the consensus label of a contradictory duplicate group.


In [ ]:

def build_soft_targets(df,hard_y):
    soft=hard_y.astype(np.float32).copy()
    if not CFG.use_soft_duplicate_targets:
        return soft
    tmp=df[["text"]+LABELS].copy()
    means=tmp.groupby("text")[LABELS].mean()
    counts=tmp.groupby("text").size()
    for i,txt in enumerate(df["text"].values):
        if counts.get(txt,1)>1:
            vals=means.loc[txt].values.astype(np.float32)
            # only soften genuine disagreement; otherwise keep the original label
            if np.any((vals>0)&(vals<1)):
                soft[i]=0.70*hard_y[i]+0.30*vals
    return soft



## 6. Training function

Each fold returns:
- OOF validation probabilities
- test probabilities

The best epoch is selected using the fold's combined F1, while thresholds are recalculated for that fold.


In [ ]:

@torch.no_grad()
def predict(model,loader):
    model.eval(); out=[]
    for b in loader:
        ids=b["input_ids"].to(DEVICE,non_blocking=True)
        mask=b["attention_mask"].to(DEVICE,non_blocking=True)
        if DEVICE.type=="cuda" and CFG.use_fp16:
            with torch.amp.autocast("cuda",dtype=torch.float16):
                lg=model(ids,mask)
        else: lg=model(ids,mask)
        out.append(torch.sigmoid(lg).float().cpu().numpy())
    return np.concatenate(out,0)

def train_fold(mc,tok,trt,trY,vat,vaY,tet,fold):
    seed_everything(CFG.seed+fold)
    model=BertBiLSTM(mc["name"],N_LABELS)
    if N_GPUS>1:
        model=nn.DataParallel(model,device_ids=list(range(N_GPUS)))
    model=model.to(DEVICE)

    trl=DataLoader(DS(trt,tok,trY),batch_size=mc["batch"],shuffle=True,
                   num_workers=2,pin_memory=True,persistent_workers=True)
    vl=DataLoader(DS(vat,tok,vaY),batch_size=64,shuffle=False,
                  num_workers=2,pin_memory=True,persistent_workers=True)
    el=DataLoader(DS(tet,tok),batch_size=64,shuffle=False,
                  num_workers=2,pin_memory=True,persistent_workers=True)

    pw=torch.tensor(pos_weight_from(trY),dtype=torch.float32,device=DEVICE)
    crit=FocalLoss(CFG.focal_alpha,CFG.focal_gamma,pw)
    opt=AdamW(model.parameters(),lr=mc["lr"],weight_decay=CFG.weight_decay)
    updates=max(1,int(np.ceil(len(trl)/mc["accum"])))
    total=updates*mc["epochs"]
    sch=get_linear_schedule_with_warmup(opt,int(total*CFG.warmup_ratio),total)
    scaler=torch.amp.GradScaler("cuda",enabled=True) if DEVICE.type=="cuda" and CFG.use_fp16 else None

    best=-1; best_val=None; best_test=None
    for ep in range(mc["epochs"]):
        model.train(); run=0.; opt.zero_grad(set_to_none=True)
        for st,b in enumerate(trl):
            ids=b["input_ids"].to(DEVICE,non_blocking=True)
            mask=b["attention_mask"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)
            if scaler:
                with torch.amp.autocast("cuda",dtype=torch.float16):
                    loss=crit(model(ids,mask),y)/mc["accum"]
                scaler.scale(loss).backward()
            else:
                loss=crit(model(ids,mask),y)/mc["accum"]; loss.backward()
            run+=loss.item()*mc["accum"]
            if (st+1)%mc["accum"]==0 or (st+1)==len(trl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                if scaler: scaler.step(opt); scaler.update()
                else: opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)

        vp=predict(model,vl)
        th=tune_thresholds(vaY,vp)
        cp=gate_preds(vp,th,True)
        score,A,B=combined_score(vaY,cp)
        print(f"{mc['name'].split('/')[-1]} fold={fold} ep={ep+1}/{mc['epochs']} loss={run/len(trl):.4f} val={score:.4f} A={A:.3f} B={B:.3f}")
        if score>best:
            best=score; best_val=vp.copy(); best_test=predict(model,el)

    del model,trl,vl,el,opt,sch,scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_val,best_test



## 7. Multilabel 5-fold OOF

Both BanglaBERT base and large use the same multilabel stratification. This gives us a cleaner estimate of generalization than selecting a blend from a single validation split.


In [ ]:

X=train_df["clean"].values
Y=train_df[LABELS].values.astype(float)
test_texts=test_df["clean"].values

splitter=MultilabelStratifiedKFold(n_splits=5,shuffle=True,random_state=42)
folds=list(splitter.split(X,Y.astype(int)))

oofs=[]; tests=[]

for mc in MODELS:
    print("\n"+"#"*70)
    print(mc["name"])
    tok=AutoTokenizer.from_pretrained(mc["name"])
    oof=np.zeros((len(train_df),N_LABELS),dtype=np.float32)
    test_model=np.zeros((len(test_df),N_LABELS),dtype=np.float32)

    for fold,(tr_idx,va_idx) in enumerate(folds):
        trY=build_soft_targets(train_df.iloc[tr_idx],Y[tr_idx])
        vp,tp=train_fold(mc,tok,X[tr_idx],trY,X[va_idx],Y[va_idx],test_texts,fold)
        oof[va_idx]=vp
        test_model+=tp/len(folds)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    oofs.append(oof); tests.append(test_model)
    th=tune_thresholds(Y,oof)
    c,A,B=combined_score(Y,gate_preds(oof,th,True))
    print(f"MODEL OOF {mc['name']}: combined={c:.5f} A={A:.5f} B={B:.5f}")



## 8. Cheap lexical specialist

Character n-grams can capture:
- spelling variants;
- Banglish;
- profanity obfuscation;
- repeated character patterns;
- slang.

It is not automatically included. Its weight is selected from OOF predictions.


In [ ]:

def tfidf_fit_predict(train_text,y,pred_text):
    word=TfidfVectorizer(analyzer="word",ngram_range=(1,2),min_df=1,max_features=70000,sublinear_tf=True)
    char=TfidfVectorizer(analyzer="char",ngram_range=(2,6),min_df=2,max_features=120000,sublinear_tf=True)
    A=word.fit_transform(train_text); B=char.fit_transform(train_text)
    P1=word.transform(pred_text); P2=char.transform(pred_text)
    Xtr=hstack([A,B],format="csr"); Xp=hstack([P1,P2],format="csr")
    out=np.zeros((len(pred_text),N_LABELS),dtype=np.float32)
    for i in range(N_LABELS):
        clf=LogisticRegression(C=1.5,class_weight="balanced",solver="liblinear",max_iter=1500)
        clf.fit(Xtr,y[:,i].astype(int))
        out[:,i]=clf.predict_proba(Xp)[:,1]
    return out

tf_oof=np.zeros((len(train_df),N_LABELS),dtype=np.float32)
for fold,(tr_idx,va_idx) in enumerate(folds):
    tf_oof[va_idx]=tfidf_fit_predict(
        train_df["raw"].values[tr_idx]+" "+train_df["clean"].values[tr_idx],
        Y[tr_idx],
        train_df["raw"].values[va_idx]+" "+train_df["clean"].values[va_idx]
    )
th=tune_thresholds(Y,tf_oof)
print("TFIDF OOF:",combined_score(Y,gate_preds(tf_oof,th,True)))



## 9. OOF ensemble search

We search the base/large/TF-IDF mixture and gate threshold using only OOF predictions.

The selected mixture is therefore evidence-based rather than chosen because a leaderboard submission happened to move up or down.


In [ ]:

# Grid over transformer mixture + lexical weight.
best=(-1,None,None,None)
for wbase in np.arange(0,1.01,0.05):
    for wtf in np.arange(0,0.41,0.05):
        wtf=min(wtf,0.40)
        remaining=1-wtf
        wlarge=remaining-wbase*remaining
        wb=remaining-wlarge
        # normalized weights: wb = base share, wl = large share
        p=wb*oofs[0]+wlarge*oofs[1]+wtf*tf_oof
        th=tune_thresholds(Y,p)
        for gate in [0.0,0.4,0.5,0.6,0.7,0.8]:
            pred=(p>=th.reshape(1,-1)).astype(int)
            if gate>0: pred[p[:,0]<gate,1:]=0
            s,A,B=combined_score(Y,pred)
            if s>best[0]:
                best=(s,(wb,wlarge,wtf),th.copy(),gate)

best_score,bweights,bthr,bgate=best
print("BEST OOF:",best_score)
print("weights base/large/tfidf:",bweights)
print("thresholds:",np.round(bthr,3))
print("gate:",bgate)



## 10. External validation sanity check

The validation set is not used to choose the final hyperparameters above.

We use it to detect whether the OOF improvement is plausible outside the training folds. This cell uses the fold ensemble's predictions on validation only if those predictions are generated in the optional final-inference section.

Because the main CV loop is deliberately optimized for runtime, the validation check below is a lightweight diagnostic hook rather than a second hidden tuning loop.


In [ ]:

# Per-label OOF error analysis.
oof_blend=best[1][0]*oofs[0]+best[1][1]*oofs[1]+best[1][2]*tf_oof
op=(oof_blend>=bthr.reshape(1,-1)).astype(int)
if bgate>0: op[oof_blend[:,0]<bgate,1:]=0

rows=[]
for i,lab in enumerate(LABELS):
    for j in np.where(op[:,i]!=Y[:,i])[0]:
        rows.append({
            "index":int(j),
            "label":lab,
            "text":train_df.iloc[j]["text"],
            "true":int(Y[j,i]),
            "prob":float(oof_blend[j,i]),
            "threshold":float(bthr[i]),
            "error":"FN" if Y[j,i]==1 else "FP"
        })
err=pd.DataFrame(rows)
err.to_csv("v12_oof_error_analysis.csv",index=False)
print("Saved v12_oof_error_analysis.csv")
print(err.groupby(["label","error"]).size())



## 11. Final training on TRAIN + VALIDATION

Once the OOF configuration is fixed, this section retrains the selected transformer models on all labeled rows.

The final test probabilities use the OOF-selected weights and thresholds.


In [ ]:

# Combine labeled data only after all OOF choices are fixed.
full_df=pd.concat([train_df,val_df],ignore_index=True)
FULL_Y=full_df[LABELS].values.astype(float)
FULL_TEXT=full_df["clean"].values
TEST_TEXT=test_df["clean"].values

# To keep the final run practical, train the two proven transformer families.
# Five folds are retained in the selection stage; final training uses 1 full-data fit per model.
final_tests=[]

def train_full(mc,texts,y,test_texts):
    tok=AutoTokenizer.from_pretrained(mc["name"])
    model=BertBiLSTM(mc["name"],N_LABELS)
    if N_GPUS>1: model=nn.DataParallel(model,device_ids=list(range(N_GPUS)))
    model=model.to(DEVICE)

    trl=DataLoader(DS(texts,tok,y),batch_size=mc["batch"],shuffle=True,num_workers=2,pin_memory=True,persistent_workers=True)
    el=DataLoader(DS(test_texts,tok),batch_size=64,shuffle=False,num_workers=2,pin_memory=True,persistent_workers=True)

    pw=torch.tensor(pos_weight_from(y),dtype=torch.float32,device=DEVICE)
    crit=FocalLoss(CFG.focal_alpha,CFG.focal_gamma,pw)
    opt=AdamW(model.parameters(),lr=mc["lr"],weight_decay=CFG.weight_decay)
    total=int(np.ceil(len(trl)/mc["accum"]))*mc["epochs"]
    sch=get_linear_schedule_with_warmup(opt,int(total*CFG.warmup_ratio),total)
    scaler=torch.amp.GradScaler("cuda",enabled=True) if DEVICE.type=="cuda" and CFG.use_fp16 else None

    for ep in range(mc["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for st,b in enumerate(trl):
            ids=b["input_ids"].to(DEVICE,non_blocking=True); mask=b["attention_mask"].to(DEVICE,non_blocking=True)
            yy=b["labels"].to(DEVICE,non_blocking=True)
            if scaler:
                with torch.amp.autocast("cuda",dtype=torch.float16): loss=crit(model(ids,mask),yy)/mc["accum"]
                scaler.scale(loss).backward()
            else:
                loss=crit(model(ids,mask),yy)/mc["accum"]; loss.backward()
            if (st+1)%mc["accum"]==0 or st+1==len(trl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                if scaler: scaler.step(opt); scaler.update()
                else: opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
        print(mc["name"],"full epoch",ep+1,"/",mc["epochs"])

    tp=predict(model,el)
    del model,trl,el,opt,sch,scaler,tok
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return tp

for mc in MODELS:
    final_tests.append(train_full(mc,FULL_TEXT,FULL_Y,TEST_TEXT))


In [ ]:

# Final TF-IDF on all labeled rows.
tf_final=tfidf_fit_predict(
    full_df["raw"].values+" "+full_df["clean"].values,
    FULL_Y,
    test_df["raw"].values+" "+test_df["clean"].values
)

final_prob=bweights[0]*final_tests[0]+bweights[1]*final_tests[1]+bweights[2]*tf_final
final_pred=(final_prob>=bthr.reshape(1,-1)).astype(int)
if bgate>0:
    final_pred[final_prob[:,0]<bgate,1:]=0

sub=pd.DataFrame({"ID":test_df["ID"].values})
for i,l in enumerate(LABELS): sub[l]=final_pred[:,i].astype(int)
sub.to_csv("submission_v12_targeted_upgrade.csv",index=False)

np.save("v12_test_prob.npy",final_prob)
np.save("v12_thresholds.npy",bthr)
np.save("v12_weights.npy",np.asarray(bweights))
np.save("v12_gate.npy",np.asarray([bgate]))

print("Saved submission_v12_targeted_upgrade.csv")
print("positive rates:")
print(sub[LABELS].mean().round(4))
print(sub.head())



# What to compare after the run

Record these four numbers:

1. **Base BanglaBERT OOF**
2. **BanglaBERT-large OOF**
3. **Best ensemble OOF**
4. **Kaggle public score**

Also send the printed per-label OOF/error analysis.

### If the score is still below 0.61

Do not immediately add more models.

The error-analysis CSV will tell us which labels are responsible. The likely next experiment would then be targeted at the weakest attribute(s), rather than increasing architecture complexity everywhere.
